# Cuaderno 4 · Meteorología en cualquier punto teniendo en cuenta el relieve

**Objetivo:** decidir, con datos, cómo estimar la meteorología (temperatura máxima, humedad y viento) de una parcela que **no tiene una estación de AEMET** en su centro.

**Por qué importa:** en Canarias la altitud cambia mucho la meteorología en pocos kilómetros. Una parcela del monte a 1.500 m puede tener como estación más cercana una a 600 m, con un clima muy distinto. Además, la **inversión térmica de los alisios** hace que, a cierta altura, la temperatura deje de bajar e incluso suba, y que la humedad caiga de golpe por encima del mar de nubes.

**Métodos que se comparan:**

| Método | Qué hace |
|---|---|
| `cercana` | Copia el valor de la estación más cercana de la isla (referencia, sin corrección) |
| `gradiente_fijo` | Estación más cercana corregida con el gradiente estándar de −6,5 °C/km; la humedad se recalcula manteniendo el punto de rocío |
| `perfil_dia` | Perfil vertical de ese día en esa isla, ajustado con sus estaciones: una recta o una recta con un **codo** en la altura de la inversión. Se evalúa en la altitud de la parcela |
| `perfil_residuos` | El perfil del día, corregido con lo que miden las estaciones vecinas (interpolación de los residuos por distancia inversa) |

**Cómo se mide:** se esconde cada estación, cada día, y se estima con las demás estaciones **de su misma isla**. Como sabemos lo que midió de verdad, calculamos el error de cada método, en especial en las estaciones de altura (≥ 1.000 m).

**Entrada:** `meteo_diaria_estaciones.csv` (Cuaderno 1, apartado 5b). No necesita Earth Engine.

## 1. Entorno

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # Clono el repositorio para usar las funciones del paquete del proyecto (src/wildfire)
    if not os.path.exists("/content/repo"):
        !git clone -q https://github.com/joseandres03/sentinel2-meteo_IA_wildfire_risk.git /content/repo
    sys.path.insert(0, "/content/repo/src")
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_CARPETA = "/content/drive/MyDrive/TFM_Riesgo_Incendios_Canarias"
else:
    sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
    RUTA_CARPETA = os.environ.get("RUTA_CARPETA", "datos")

from wildfire import meteo_relieve as mr
from wildfire.validacion import isla_de

RUTA_METEO = os.path.join(RUTA_CARPETA, "meteo_diaria_estaciones.csv")
RUTA_RESULTADOS = os.path.join(RUTA_CARPETA, "meteorologia_relieve")
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

MIN_DIAS_ESTACION = 365   # descarto estaciones con menos de un año de datos
PASO_DIAS = 1             # 1 = todos los días; 2 o 3 acelera la validación usando 1 de cada N días
UMBRAL_ALTURA = 1000      # m, separa estaciones "de costa y medianías" y "de altura"

## 2. Datos y cobertura de las estaciones

No todas las estaciones cubren los 10 años: algunas empezaron más tarde o tienen huecos. Miro cuántos días tiene cada una y descarto las que tienen menos de un año.

In [ ]:
meteo = pd.read_csv(RUTA_METEO, parse_dates=["fecha"])
meteo = meteo.rename(columns={"latitud_dec": "lat", "longitud_dec": "lon"})
meteo["isla"] = isla_de(meteo["lat"], meteo["lon"])

dias = meteo.groupby("indicativo").agg(nombre=("nombre", "first"), isla=("isla", "first"),
                                        altitud=("altitud", "first"), dias=("fecha", "nunique"),
                                        desde=("fecha", "min"), hasta=("fecha", "max"))
descartadas = dias[dias["dias"] < MIN_DIAS_ESTACION]
print(f"Estaciones: {len(dias)} · descartadas por tener menos de {MIN_DIAS_ESTACION} días: {len(descartadas)}")
if len(descartadas):
    print(descartadas[["nombre", "isla", "dias"]].to_string())

meteo = meteo[~meteo["indicativo"].isin(descartadas.index)].reset_index(drop=True)
dias = dias.drop(descartadas.index)

resumen_islas = dias.groupby("isla").agg(estaciones=("dias", "size"), altitud_min=("altitud", "min"),
                                         altitud_max=("altitud", "max"), dias_medios=("dias", "mean")).round(0)
resumen_islas

In [ ]:
# Cobertura temporal: una línea por estación, ordenadas por isla y altitud
orden = dias.sort_values(["isla", "altitud"])
fig, ax = plt.subplots(figsize=(10, 12))
for k, (ind, f) in enumerate(orden.iterrows()):
    fechas = meteo.loc[meteo["indicativo"] == ind, "fecha"]
    ax.plot(fechas, np.full(len(fechas), k), "|", ms=3, color="tab:blue", alpha=0.6)
ax.set_yticks(range(len(orden)))
ax.set_yticklabels([f"{f.isla[:3]} · {f.nombre[:22]} ({f.altitud:.0f} m)" for f in orden.itertuples()], fontsize=6)
ax.set_title("Días con dato por estación")
plt.tight_layout(); plt.show()

## 3. Un día de ejemplo: el perfil vertical

Para entender el método, tomo el día de verano con más estaciones en Tenerife y represento la temperatura máxima de cada estación frente a su altitud, con el perfil ajustado. Si el ajuste encuentra un **codo**, esa es la altura estimada de la inversión ese día.

In [ ]:
tf = meteo[(meteo["isla"] == "Tenerife") & meteo["fecha"].dt.month.isin([7, 8]) & meteo["tmax"].notna()]
dia = tf.groupby("fecha").size().idxmax()
g = tf[tf["fecha"] == dia]

perfil = mr.ajustar_perfil(g["altitud"], g["tmax"])
z = np.linspace(0, g["altitud"].max() + 100, 200)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(g["tmax"], g["altitud"], color="tab:red", zorder=3, label="estaciones")
ax.plot(mr.evaluar_perfil(perfil, z), z, "k-", label=f"perfil ajustado ({perfil['tipo']})")
ax.plot(g["tmax"].max() + mr.GRADIENTE_ESTANDAR * (z - g.loc[g["tmax"].idxmax(), "altitud"]), z, "k:",
        label="gradiente estándar −6,5 °C/km")
if perfil["z_codo"]:
    ax.axhline(perfil["z_codo"], color="tab:blue", ls="--", label=f"codo (inversión) ≈ {perfil['z_codo']:.0f} m")
for r in g.itertuples():
    ax.annotate(r.nombre[:14], (r.tmax, r.altitud), fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.set(xlabel="Temperatura máxima (°C)", ylabel="Altitud (m)", title=f"Tenerife · {dia:%Y-%m-%d}")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 4. La inversión, día a día

Ajusto el perfil de temperatura máxima cada día en cada isla con todas sus estaciones y cuento cuántos días aparece un codo y a qué altura. En islas con pocas estaciones de altura (Lanzarote, Fuerteventura) el método no debería encontrar codos: no hay datos para verlos.

In [ ]:
perfiles = mr.perfiles_diarios(meteo, "tmax")
perfiles["mes"] = pd.to_datetime(perfiles["fecha"]).dt.month

resumen_codo = perfiles.groupby("isla").agg(
    dias=("tipo", "size"),
    porcentaje_con_codo=("tipo", lambda t: round(100 * (t == "codo").mean(), 1)),
    altura_codo_mediana=("z_codo", "median"),
    gradiente_mediano_por_km=("pendiente_por_km", "median"))
resumen_codo.round(2)

In [ ]:
con_codo = perfiles[perfiles["tipo"] == "codo"]
islas_codo = [i for i in ("Tenerife", "Gran Canaria", "La Palma", "La Gomera", "El Hierro")
              if (con_codo["isla"] == i).sum() > 30]
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
(perfiles.assign(codo=perfiles["tipo"] == "codo").groupby(["mes", "isla"])["codo"].mean().unstack()[islas_codo] * 100
 ).plot(ax=ejes[0], marker="o")
ejes[0].set(title="Días con inversión detectada (%)", xlabel="Mes", ylabel="%")
con_codo[con_codo["isla"].isin(islas_codo)].groupby(["mes", "isla"])["z_codo"].median().unstack().plot(ax=ejes[1], marker="o")
ejes[1].set(title="Altura mediana de la inversión", xlabel="Mes", ylabel="m")
plt.tight_layout(); plt.show()

## 5. Validación dejando una estación fuera

Para cada día y cada estación: la escondo, la estimo con las demás de su isla con los cuatro métodos y guardo el valor real y el estimado. Variables: temperatura máxima (`tmax`, °C), humedad relativa media (`hrMedia`, %) y viento medio (`velmedia`, m/s).

Con todos los días tarda varios minutos; `PASO_DIAS = 2` o `3` lo acelera. Los resultados se guardan en Drive para no repetirlo.

In [ ]:
ruta_res = os.path.join(RUTA_RESULTADOS, f"validacion_paso{PASO_DIAS}.csv")
if os.path.exists(ruta_res):
    res = pd.read_csv(ruta_res, parse_dates=["fecha"])
    print("Resultados cargados de Drive")
else:
    t0 = time.time()
    res = mr.validacion_dejando_una_fuera(meteo, paso_dias=PASO_DIAS)
    res.to_csv(ruta_res, index=False)
    print(f"Validación completada en {(time.time() - t0) / 60:.1f} min")
print(f"{len(res):,} estimaciones")

## 6. Resultados: ¿qué método se equivoca menos?

- **MAE:** error absoluto medio (en °C, % o m/s).
- **Sesgo:** error medio con signo. Positivo = el método sobrestima.

In [ ]:
tabla = mr.resumen_errores(res, UMBRAL_ALTURA)
mae = tabla.pivot_table(index=["variable", "franja"], columns="metodo", values="MAE")
mae = mae[[m for m in mr.METODOS if m in mae.columns]]
mae.style.format("{:.2f}").highlight_min(axis=1, color="#d9f2d9")

In [ ]:
sesgo = tabla.pivot_table(index=["variable", "franja"], columns="metodo", values="sesgo")
sesgo[[m for m in mr.METODOS if m in sesgo.columns]].round(2)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (var, unidad) in zip(ejes, [("tmax", "°C"), ("hrMedia", "%"), ("velmedia", "m/s")]):
    t = tabla[(tabla["variable"] == var) & (tabla["franja"] != "todas")]
    t.pivot(index="metodo", columns="franja", values="MAE").reindex(
        [m for m in mr.METODOS if m in set(t["metodo"])]).plot.bar(ax=ax, rot=20)
    ax.set(title=f"MAE · {var}", ylabel=unidad, xlabel="")
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, "mae_por_metodo.png"), dpi=150)
plt.show()

### La prueba más dura: estaciones de altura

Error de temperatura máxima y de humedad en cada estación por encima de 1.000 m. Aquí se ve si algún método estima bien, por ejemplo, Izaña o Las Cañadas sin usarlas.

In [ ]:
altas = res[res["altitud"] >= UMBRAL_ALTURA]
por_estacion = (altas.groupby(["variable", "indicativo", "metodo"])["error"].apply(lambda e: e.abs().mean())
                .unstack("metodo")[list(mr.METODOS)])
por_estacion = por_estacion.join(dias[["nombre", "isla", "altitud"]], on="indicativo")
for var in ("tmax", "hrMedia"):
    print(f"\nMAE por estación de altura · {var}")
    print(por_estacion.loc[var].sort_values("altitud", ascending=False).round(2).to_string())

### Por isla

In [ ]:
(res[res["variable"] == "tmax"].groupby(["isla", "metodo"])["error"].apply(lambda e: e.abs().mean())
 .unstack()[list(mr.METODOS)].round(2))

## 7. Conclusiones

> Rellenar tras ejecutar con los datos reales:
>
> - Qué método tiene menor error para cada variable, en general y en las estaciones de altura.
> - Cuánto mejora respecto a copiar la estación más cercana.
> - Si la inversión detectada tiene sentido físico (porcentaje de días, altura y estacionalidad).
> - Qué método se usará para asignar la meteorología a las parcelas del dataset de incendios.